# Flood assistant: base → SFT → DPO (Colab)

One notebook, four parts. Every part loads its own fresh model, so you can re-run any part safely.

**How to run it**
1. Run **0. Setup** once per session (install, login, Drive).
2. In **1. Config**, set `SIZE` and keep `SMOKE = True`. Run Parts A, B, C quickly to check nothing crashes (5 training steps each).
3. Set `SMOKE = False`, re-run **1. Config**, then run Parts A, B, C again for the real results.

Upload these files to `/content` first: `train.jsonl`, `test.jsonl` (from `data/splits/`) and `dpo_train_REJECTED_UNSAFE.jsonl` (from `data/dpo/`).
Everything is saved to Google Drive, so a session reset does not lose your adapters.

## 0. Setup (once per session)

In [ ]:
!pip install -q peft trl
!pip uninstall -y -q torchao

In [ ]:
from google.colab import userdata, drive
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))   # needs the HF_TOKEN secret, with notebook access on
drive.mount("/content/drive")

In [ ]:
!nvidia-smi

## 1. Config (the only cell you edit)

In [ ]:
import os, json, gc, torch

SIZE  = "3b"     # "1b" or "3b"
SMOKE = True     # True = 5-step test run.  False = the real run.   <-- CHANGE THIS, then re-run this cell

MODELS = {"1b": "meta-llama/Llama-3.2-1B-Instruct", "3b": "meta-llama/Llama-3.2-3B-Instruct"}
MODEL  = MODELS[SIZE]

# (per-device batch, gradient accumulation, gradient checkpointing). Effective batch = 16 in every case.
SFT_BATCH = {"1b": (4, 4, False), "3b": (2, 8, True)}[SIZE]
DPO_BATCH = {"1b": (2, 8, False), "3b": (1, 16, True)}[SIZE]

STEPS = 5 if SMOKE else -1
TAG   = f"{SIZE}_smoke" if SMOKE else SIZE          # smoke runs get their own names, so they can never be mistaken for real ones

OUT     = f"/content/drive/MyDrive/flood_{SIZE}"     # survives a session reset
SFT_DIR = f"{OUT}/sft_adapter_{TAG}"
DPO_DIR = f"{OUT}/dpo_adapter_{TAG}"
os.makedirs(OUT, exist_ok=True)

LORA = dict(r=8, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

print(f"model: {MODEL}\nsmoke test: {SMOKE}  (steps = {STEPS})\nsaving to: {OUT}")
if SMOKE:
    print("\n*** SMOKE TEST: the adapters from this run are NOT trained. Set SMOKE = False for real results. ***")
for f in ("train.jsonl", "test.jsonl", "dpo_train_REJECTED_UNSAFE.jsonl"):
    print(("ok      " if os.path.exists(f) else "MISSING ") + f)

## 2. Helpers (run once, no edits)

In [ ]:
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
TEST_ROWS = [json.loads(l) for l in open("test.jsonl")]
SYSTEM_PROMPT = TEST_ROWS[0]["messages"][0]["content"]   # the exact system prompt used in training and testing


def free_gpu(*names):
    """Delete old models/trainers so two copies never sit in GPU memory."""
    for n in names:
        globals().pop(n, None)
    gc.collect(); torch.cuda.empty_cache()


def load_base():
    return AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")


def get_sft_model():
    """Base model + the SFT adapter, merged. This is the SFT model (and the start point for DPO)."""
    return PeftModel.from_pretrained(load_base(), SFT_DIR).merge_and_unload()


def ask(model, question, max_new=350):
    """One quick question, with the standard system prompt."""
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


def run49(model, name, bs=8, max_new=512):
    """Answer all 49 test questions (greedy). Saves <name>.json to Drive."""
    model.eval(); results = []
    tokenizer.padding_side = "left"
    try:
        for i in range(0, len(TEST_ROWS), bs):
            batch = TEST_ROWS[i:i + bs]
            prompts = [tokenizer.apply_chat_template(r["messages"][:2], add_generation_prompt=True, tokenize=False) for r in batch]
            enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False)
            for r, seq in zip(batch, gen):
                results.append({"question": r["messages"][1]["content"], "urgency": r["labels"]["urgency"],
                                "answer": tokenizer.decode(seq[enc["input_ids"].shape[1]:], skip_special_tokens=True)})
    finally:
        tokenizer.padding_side = "right"
    path = f"{OUT}/test49_{name}.json"
    json.dump(results, open(path, "w"))
    w = [len(r["answer"].split()) for r in results]
    print(f"saved {path}\n{len(results)} answers | avg {sum(w)/len(w):.0f} words | mention 911: {sum('911' in r['answer'] for r in results)}/{len(results)}")
    return results


def series(log, key):
    pts = [(r["step"], r[key]) for r in log if key in r]
    return list(zip(*pts)) if pts else ([], [])


def plot_sft(trainer, name):
    log = trainer.state.log_history
    plt.figure(figsize=(7, 4))
    plt.plot(*series(log, "loss"), label="train")
    plt.plot(*series(log, "eval_loss"), "o-", label="validation")
    plt.xlabel("step"); plt.ylabel("loss"); plt.title(f"SFT loss ({name})"); plt.legend(); plt.grid(alpha=0.3)
    plt.savefig(f"{OUT}/sft_loss_{name}.png", dpi=150, bbox_inches="tight"); plt.show()


def plot_dpo(trainer, name):
    log = trainer.state.log_history
    fig, ax = plt.subplots(2, 2, figsize=(11, 7))
    for a, (title, tr, ev) in zip(ax.flat, [("DPO loss", "loss", "eval_loss"),
                                           ("Reward accuracy", "rewards/accuracies", "eval_rewards/accuracies"),
                                           ("Reward margin", "rewards/margins", "eval_rewards/margins")]):
        a.plot(*series(log, tr), label="train"); a.plot(*series(log, ev), "o-", label="validation"); a.set_title(title); a.legend()
    ax[1, 1].plot(*series(log, "logps/chosen"), label="chosen"); ax[1, 1].plot(*series(log, "logps/rejected"), label="rejected")
    ax[1, 1].set_title("Log-probability"); ax[1, 1].legend()
    for a in ax.flat:
        a.set_xlabel("step"); a.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{OUT}/dpo_curves_{name}.png", dpi=150); plt.show()

## Part A. Base model: load and test
No training here. This is the "before" number.

In [ ]:
free_gpu("model", "trainer", "base", "sft_model", "dpo_trainer")
model = load_base()
print(f"loaded {MODEL}: {sum(p.numel() for p in model.parameters())/1e9:.2f}B parameters")

In [ ]:
# quick sanity check on a few questions
ask(model, "water up to my knees in basement, stairs half under, what do i do")
print("\n" + "=" * 60 + "\n")
ask(model, "walking back home w my neighbor who hurt his ankle, can we step over that downed powerline if its not sparking")

In [ ]:
# all 49 test questions
res_base = run49(model, f"base-{SIZE}")

## Part B. SFT (supervised fine-tuning with LoRA): train, then test
Full run = 3 epochs = about 78 steps. The save cell prints the step count, so you can see at once if it was only the smoke test.

In [ ]:
# data: 90% train / 10% validation of train.jsonl. The loss is computed on the answer only.
def to_prompt_completion(row):
    m = row["messages"]
    return {"prompt": m[:2], "completion": m[2:]}

_ds = load_dataset("json", data_files="train.jsonl", split="train").train_test_split(test_size=0.1, seed=42)
sft_train = _ds["train"].map(to_prompt_completion, remove_columns=["messages"])
sft_val   = _ds["test"].map(to_prompt_completion, remove_columns=["messages"])
print(len(sft_train), "train /", len(sft_val), "validation")

In [ ]:
from trl import SFTConfig, SFTTrainer

free_gpu("model", "trainer", "base", "sft_model", "dpo_trainer")
model = load_base()

bs, accum, ckpt = SFT_BATCH
sft_cfg = SFTConfig(
    output_dir="lora_out",
    num_train_epochs=3,
    max_steps=STEPS,
    per_device_train_batch_size=bs,
    gradient_accumulation_steps=accum,
    gradient_checkpointing=ckpt,
    gradient_checkpointing_kwargs={"use_reentrant": False} if ckpt else None,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=4,
    fp16=True,
    logging_steps=5,
    eval_strategy="steps" if STEPS > 0 else "epoch",
    eval_steps=5,
    save_strategy="no",
    max_length=1024,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=sft_cfg, train_dataset=sft_train, eval_dataset=sft_val,
                     processing_class=tokenizer, peft_config=LoraConfig(**LORA))
trainer.train()

In [ ]:
# save the adapter + loss curve
steps_done = trainer.state.global_step
trainer.save_model(SFT_DIR); tokenizer.save_pretrained(SFT_DIR)
plot_sft(trainer, TAG)
print(f"saved {SFT_DIR}\ntrained steps: {steps_done}   (a full run is about 78)")
if SMOKE or steps_done < 70:
    print("\n*** This adapter is NOT fully trained. Do not use it for results. ***")

In [ ]:
# test the SFT model: base + adapter, merged
free_gpu("model", "trainer", "base", "sft_model", "dpo_trainer")
sft_model = get_sft_model()
ask(sft_model, "walking back home w my neighbor who hurt his ankle, can we step over that downed powerline if its not sparking")
res_sft = run49(sft_model, f"sft-{TAG}")

## Part C. DPO (learn from good vs bad answers): train, then test
Starts from the **merged SFT model**, so the reference model (adapter off) is the SFT model.
Needs the SFT adapter from Part B (full run). Full run = 2 epochs = about 52 steps.

In [ ]:
# data: 452 pairs (prompt / chosen / rejected), 90% train / 10% validation
_dpo = load_dataset("json", data_files="dpo_train_REJECTED_UNSAFE.jsonl", split="train").train_test_split(test_size=0.1, seed=42)
dpo_train, dpo_val = _dpo["train"], _dpo["test"]
print(len(dpo_train), "train /", len(dpo_val), "validation")

In [ ]:
from trl import DPOConfig, DPOTrainer

free_gpu("model", "trainer", "base", "sft_model", "dpo_trainer")
sft_model = get_sft_model()          # loads SFT_DIR; if it does not exist, run Part B first

bs, accum, ckpt = DPO_BATCH
dpo_cfg = DPOConfig(
    output_dir="dpo_out",
    num_train_epochs=2,
    max_steps=STEPS,
    per_device_train_batch_size=bs,
    gradient_accumulation_steps=accum,
    gradient_checkpointing=ckpt,
    gradient_checkpointing_kwargs={"use_reentrant": False} if ckpt else None,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=3,
    beta=0.1,
    fp16=True,
    logging_steps=1 if SMOKE else 5,
    eval_strategy="steps" if STEPS > 0 else "epoch",
    eval_steps=5,
    save_strategy="no",
    max_length=1024,
    report_to="none",
)
dpo_trainer = DPOTrainer(model=sft_model, args=dpo_cfg, train_dataset=dpo_train, eval_dataset=dpo_val,
                         processing_class=tokenizer, peft_config=LoraConfig(**LORA))
dpo_trainer.train()

In [ ]:
# save the adapter + curves
steps_done = dpo_trainer.state.global_step
dpo_trainer.save_model(DPO_DIR)
plot_dpo(dpo_trainer, TAG)
print(f"saved {DPO_DIR}\ntrained steps: {steps_done}   (a full run is about 52)")
if SMOKE or steps_done < 45:
    print("\n*** This adapter is NOT fully trained. Do not use it for results. ***")

In [ ]:
# test the DPO model
ask(dpo_trainer.model, "walking back home w my neighbor who hurt his ankle, can we step over that downed powerline if its not sparking")
res_dpo = run49(dpo_trainer.model, f"dpo-{TAG}")

## 3. Results and files

In [ ]:
# compare every results file saved so far
import glob
for p in sorted(glob.glob(f"{OUT}/test49_*.json")):
    r = json.load(open(p)); w = [len(x["answer"].split()) for x in r]
    print(f"{os.path.basename(p):34} {len(r)} answers | avg {sum(w)/len(w):4.0f} words | 911: {sum('911' in x['answer'] for x in r)}/{len(r)}")
print("\nfiles on Drive:"); print("\n".join(sorted(os.listdir(OUT))))

In [ ]:
# optional: save the merged DPO model to Drive (about 2.5 GB for 1B, 6.4 GB for 3B) for GGUF conversion on the Mac
SAVE_MERGED = False
if SAVE_MERGED:
    merged = dpo_trainer.model.merge_and_unload()
    merged.save_pretrained(f"{OUT}/dpo_merged_{TAG}"); tokenizer.save_pretrained(f"{OUT}/dpo_merged_{TAG}")
    print("saved merged model")